In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tauCC.src.fairness_metrics import balance_gen

In [2]:
def create_path(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [3]:
root = os.getcwd()

In [4]:
"""
sensitive = np.load(root + "/datasets/synthetic/clus5/groups2/sensitive_p1.0.npy")
matrix = np.load(root + "/datasets/synthetic/clus5/matrix.npy")
row_labels = np.load(root + "/datasets/synthetic/clus5/row_labels.npy")
col_labels = np.load(root + "/datasets/synthetic/clus5/col_labels.npy")
balance_row = balance_gen(sensitive, row_labels)
balance_row
"""

'\nsensitive = np.load(root + "/datasets/synthetic/clus5/groups2/sensitive_p1.0.npy")\nmatrix = np.load(root + "/datasets/synthetic/clus5/matrix.npy")\nrow_labels = np.load(root + "/datasets/synthetic/clus5/row_labels.npy")\ncol_labels = np.load(root + "/datasets/synthetic/clus5/col_labels.npy")\nbalance_row = balance_gen(sensitive, row_labels)\nbalance_row\n'

In [5]:
clusters = 3
groups = 3
taucc_path = root + f"/results/synthetic/clus{clusters}"

In [6]:
taucc_fair_max = pd.read_csv(taucc_path + f"/taucc_fair_max/topsis_best_param_groups{groups}.csv")
taucc_fair = pd.read_csv(taucc_path + f"/taucc_fair/topsis_best_param_groups{groups}.csv")
taucc_vanilla = pd.read_csv(taucc_path + f"/taucc_vanilla/aggregated_groups{groups}.csv")

In [7]:
plot_path = root + f"/plots/synthetic"
create_path(plot_path)
plot_path += f"/clus{clusters}"
create_path(plot_path)
plot_path += f"/groups{groups}"
create_path(plot_path)

## Plot of Results

In [8]:
x = np.array(taucc_fair_max["sensitive_p"].values)
x

array([0. , 0.3, 0.5, 0.8, 1. ])

In [9]:
taucc_vanilla = taucc_vanilla[taucc_vanilla["sensitive_p"].isin(x)]
taucc_vanilla

,sensitive_p,tau_x_mean,tau_x_std,tau_x_var,tau_y_mean,tau_y_std,tau_y_var,NMI_rows_mean,NMI_rows_std,NMI_rows_var,...,balance_chierichetti_var,balance_bera_mean,balance_bera_std,balance_bera_var,KL_fairness_error_mean,KL_fairness_error_std,KL_fairness_error_var,time_mean,time_std,time_var
0,0.0,0.303141,0.013354,0.000178,0.300448,0.014774,0.000218,0.946601,0.112575,0.012673,...,0.000002,0.996233,0.000622,3.874117e-07,0.000005,0.000001,2.226871e-12,0.184702,0.231721,0.053695
3,0.3,0.309475,0.016356,0.000268,0.307455,0.018095,0.000327,0.893203,0.137876,0.019010,...,0.010592,0.764524,0.075474,5.696266e-03,0.030569,0.018371,3.374883e-04,0.164601,0.186205,0.034672
5,0.5,0.306336,0.015345,0.000235,0.303980,0.016971,0.000288,0.919800,0.129136,0.016676,...,0.003943,0.525076,0.077714,6.039539e-03,0.108827,0.030932,9.567695e-04,0.105458,0.068947,0.004754
8,0.8,0.315866,0.016405,0.000269,0.314519,0.018143,0.000329,0.839600,0.138053,0.019059,...,0.021077,0.363597,0.202432,4.097890e-02,0.263111,0.167334,2.800078e-02,0.133578,0.089652,0.008037
10,1.0,0.309475,0.016356,0.000268,0.307455,0.018095,0.000327,0.893203,0.137876,0.019010,...,0.019753,0.100100,0.211030,4.453347e-02,NaN,NaN,NaN,0.090722,0.037317,0.001393


In [10]:
def plot_metric(
    x,
    taucc_vanilla,
    taucc_fair,
    taucc_fair_max,
    metric,
    plot_path,
    metric_label,
    title,
    range_values=None
):
    # Colori palette Wong (2011) - colorblind-friendly
    COLOR_VANILLA  = "#E69F00"
    COLOR_FAIR     = "#0072B2"
    COLOR_FAIR_MAX = "#009E73"
    
    mean_col = f"{metric}_mean"
    std_col  = f"{metric}_std"

    cols = [mean_col, std_col]
    
    # Fast TauCC
    vanilla_mean = np.full(len(x), taucc_vanilla[mean_col].values)
    vanilla_std  = np.full(len(x), taucc_vanilla[std_col].values)

    # Fair TauCC v1
    fair_mean = np.array(taucc_fair[mean_col].values)
    fair_std  = np.array(taucc_fair[std_col].values)

    # Fair TauCC v2 (max)
    fair_max_mean = np.array(taucc_fair_max[mean_col].values)
    fair_max_std  = np.array(taucc_fair_max[std_col].values)

    # Plot
    fig, ax = plt.subplots()

    ax.plot(x, vanilla_mean, label="Fast $\\tau$CC", color=COLOR_VANILLA, linestyle='--', linewidth=1.5)
    ax.fill_between(x, vanilla_mean - vanilla_std, vanilla_mean + vanilla_std, alpha=0.15, color=COLOR_VANILLA)

    ax.plot(x, fair_mean, label="Fair $\\tau$CC v1", color=COLOR_FAIR, linewidth=1.5)
    ax.fill_between(x, fair_mean - fair_std, fair_mean + fair_std, alpha=0.15, color=COLOR_FAIR)

    ax.plot(x, fair_max_mean, label="Fair $\\tau$CC v2", color=COLOR_FAIR_MAX, linewidth=1.5)
    ax.fill_between(x, fair_max_mean - fair_max_std, fair_max_mean + fair_max_std, alpha=0.15, color=COLOR_FAIR_MAX)

    if range_values is not None:
        ax.set_ylim(range_values[0], range_values[1] + 0.05)
    
    if "tau_" in metric:
        all_df = [taucc_vanilla, taucc_fair, taucc_fair_max]
        max_val = max(
            np.max(df[c_mean].values + df[c_std].values)
            for df in all_df
            for c_mean, c_std in [("tau_x_mean", "tau_x_std"), ("tau_y_mean", "tau_y_std")]
        )
        ax.set_ylim(0, max_val + 0.05)
    
    ax.set_xlim(0.0, 1.0)
    #ax.set_xticks(x)
    #ax.set_xticklabels(x)
    
    ax.legend(framealpha=0.9, edgecolor='gray', fontsize=13)

    ax.set_xlabel('sensitive degree', fontsize=14)
    ax.set_ylabel(metric_label, fontsize=14)
    ax.set_title(f"{title}", fontsize=16)
    ax.tick_params(axis="both", labelsize=12)
    
    ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
    plt.tight_layout()
    plt.savefig(f"{plot_path}/{metric}.png", dpi=300)
    #plt.show()
    plt.close(fig)

In [11]:
plot_path

'/home/peiretti/fair-clustering/plots/synthetic/clus3/groups3'

In [12]:
metrics = [
    ("balance_bera",     "balance", "Balance", [0.0,1.0]),
    ("tau_x",            "tau x",   "tau x", None),
    ("tau_y",            "tau y",   "tau y", None),
    #("ARI_true_labels",  "ARI",     "ARI w.r.t. true labels", None),
    #("ARI_rows",         "ARI",     "ARI w.r.t. row clusters", None),
    #("ARI_cols",         "ARI",     "ARI w.r.t. column clusters", None),
    #("time", "time (sec)", "Execution time (in seconds)", None)
]

for metric, metric_label, title, range_values in metrics:    
    plot_metric(
        x=x,
        taucc_vanilla=taucc_vanilla,
        taucc_fair=taucc_fair,
        taucc_fair_max=taucc_fair_max,
        metric=metric,
        metric_label=metric_label,
        title=title,
        range_values=range_values,
        plot_path=plot_path,
    )

# Color map

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math

In [ ]:
def create_path(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [ ]:
root = os.getcwd()

In [ ]:
clusters = 3
groups = 2
taucc_path = root + f"/results/synthetic/clus{clusters}"

In [ ]:
taucc_fair_max = pd.read_csv(taucc_path + f"/taucc_fair_max/aggregated_groups{groups}.csv")
taucc_fair = pd.read_csv(taucc_path + f"/taucc_fair/aggregated_groups{groups}.csv")
taucc_vanilla = pd.read_csv(taucc_path + f"/taucc_vanilla/aggregated_groups{groups}.csv")

In [ ]:
heatmap_path = root + f"/plots/synthetic"
create_path(heatmap_path)
heatmap_path += f"/clus{clusters}"
create_path(heatmap_path)
heatmap_path += f"/groups{groups}"
create_path(heatmap_path)
heatmap_path += f"/heatmap"
create_path(heatmap_path)
heatmap_path

In [ ]:
def plot_heatmap(
    df,
    metric,
    algorithm,
    clusters,
    groups,
    plot_path,
    range_values=None,
    metric_label=None,
    title=None,
    col_param="fair_majority",
    row_param="fair_minority",
    sensitive_param="sensitive_p"
):
    if algorithm == "taucc_fair_max":
        algorithm_name = "Fair $\\tau$CC v2"
    elif algorithm == "taucc_fair":
        algorithm_name = "Fair $\\tau$CC v1"
    else:
        raise Exception("The algorithm does not exist. Choose an option: taucc_fair or taucc_fair_max.")

    mean_col = f"{metric}_mean"
    sensitive_values = sorted(df[sensitive_param].unique())
    n = len(sensitive_values)

    # Layout: fino a 3 colonne, poi va a capo
    ncols = min(n, 3)
    nrows = math.ceil(n / ncols)

    # Range valori comune a tutti i subplot
    if range_values is not None:
        vmin, vmax = range_values
    else:
        vmin = df[mean_col].min()
        vmax = df[mean_col].max()

    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(5 * ncols, 4.5 * nrows),
        squeeze=False
    )

    for idx, sp_val in enumerate(sensitive_values):
        row_idx = idx // ncols
        col_idx = idx % ncols
        ax = axes[row_idx][col_idx]

        sub = df[df[sensitive_param] == sp_val]
        pivot = sub.pivot(index=row_param, columns=col_param, values=mean_col)
        pivot = pivot.sort_index(ascending=False)

        # Mostra la colorbar solo nell'ultimo subplot
        show_cbar = (idx == n - 1)

        sns.heatmap(
            pivot,
            ax=ax,
            cmap="viridis",
            vmin=vmin,
            vmax=vmax,
            annot=True,
            fmt=".2f",
            annot_kws={"size": 7},
            cbar=show_cbar,
            cbar_kws={"label": metric_label if metric_label else metric} if show_cbar else {},
        )
        ax.set_xlabel(col_param)
        ax.set_ylabel(row_param)
        ax.set_title(f"sensitive degree = {sp_val:.2f}")
        ax.grid(False)

    # Nascondi gli assi vuoti se n non è multiplo di ncols
    for idx in range(n, nrows * ncols):
        axes[idx // ncols][idx % ncols].set_visible(False)

    fig.suptitle(
        f"synthetic matrix ({clusters} clusters, {groups} groups) \n {algorithm_name} - {title}",
        fontsize=13,
        y=1.01
    )

    plt.tight_layout()
    plt.savefig(f"{plot_path}/{algorithm}_{metric}.png", dpi=300, bbox_inches="tight")
    #plt.show()
    plt.close(fig)

In [ ]:
def plot_heatmap_single(
    df,
    metric,
    algorithm,
    clusters,
    groups,
    plot_path,
    range_values=None,
    metric_label=None,
    title=None,
    col_param="fair_majority",
    row_param="fair_minority",
    sensitive_param="sensitive_p",
    sensitive_value=1.0
):
    if algorithm == "taucc_fair_max":
        algorithm_name = "Fair $\\tau$CC v2"
    elif algorithm == "taucc_fair":
        algorithm_name = "Fair $\\tau$CC v1"
    else:
        raise Exception("The algorithm does not exist. Choose an option: taucc_fair or taucc_fair_max.")

    mean_col = f"{metric}_mean"

    # Filtra solo il valore sensitive desiderato
    sub = df[df[sensitive_param] == sensitive_value]
    if sub.empty:
        raise ValueError(f"No data found for {sensitive_param} == {sensitive_value}")

    pivot = sub.pivot(index=row_param, columns=col_param, values=mean_col)
    pivot = pivot.sort_index(ascending=False)

    if range_values is not None:
        vmin, vmax = range_values
    else:
        vmin = sub[mean_col].min()
        vmax = sub[mean_col].max()

    fig, ax = plt.subplots(figsize=(5, 4.5))

    sns.heatmap(
        pivot,
        ax=ax,
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
        annot=False,
        #annot=True,
        #fmt=".2f",
        #annot_kws={"size": 7},
        cbar=True,
        cbar_kws={"label": metric_label if metric_label else metric},
    )

    ax.set_xlabel(r"$\alpha$ majority", fontsize=14)
    ax.set_ylabel(r"$\alpha$ minority", fontsize=14)
    ax.tick_params(axis="both", labelsize=12)
    #ax.set_title(f"sensitive degree = {sensitive_value:.2f}", fontsize=16)
    
    ax.set_title(f"{title}", fontsize=16)
    
    
    cbar = ax.collections[0].colorbar
    cbar.set_label(metric_label if metric_label else metric, fontsize=14)
    cbar.ax.tick_params(labelsize=12)
    
    ax.grid(False)
    plt.tight_layout()
    
    plt.savefig(
        f"{plot_path}/{algorithm}_{metric}_sp{sensitive_value}.png",
        dpi=300,
        bbox_inches="tight"
    )
    
    #plt.show()
    plt.close(fig)

In [ ]:
metrics = [
    ("balance_bera",     "balance", "Balance", [0.0,1.0]),
    ("tau_x",            "tau x",   "tau x", None),
    ("tau_y",            "tau y",   "tau y", None),
    #("ARI_true_labels",  "ARI",     "ARI w.r.t. true labels", None),
    #("ARI_rows",         "ARI",     "ARI w.r.t. row clusters", None),
    #("ARI_cols",         "ARI",     "ARI w.r.t. column clusters", None),
]

for metric, metric_label, title, range_values in metrics:
    """
    plot_heatmap(
        df=taucc_fair_max,
        metric=metric,
        algorithm="taucc_fair_max",
        clusters=clusters,
        groups=groups,
        plot_path=heatmap_path,
        range_values=range_values,
        metric_label=metric_label,
        title=title,
        col_param="fair_majority",
        row_param="fair_minority",
        sensitive_param="sensitive_p"
    )
    
    plot_heatmap(
        df=taucc_fair,
        metric=metric,
        algorithm="taucc_fair",
        clusters=clusters,
        groups=groups,
        plot_path=heatmap_path,
        range_values=range_values,
        metric_label=metric_label,
        title=title,
        col_param="fair_majority",
        row_param="fair_minority",
        sensitive_param="sensitive_p"
    )
    """
    
    plot_heatmap_single(
        df=taucc_fair_max,
        metric=metric,
        algorithm="taucc_fair_max",
        clusters=clusters,
        groups=groups,
        plot_path=heatmap_path,
        range_values=range_values,
        metric_label=metric_label,
        title=title,
        col_param="fair_majority",
        row_param="fair_minority",
        sensitive_param="sensitive_p",
        sensitive_value=1.0
    )

In [ ]:
metrics = [
    ("balance_bera",     "balance", "Balance", [0.0,1.0]),
    ("tau_x",            "tau x",   "tau x", None),
    ("tau_y",            "tau y",   "tau y", None),
    ("ARI_true_labels",  "ARI",     "ARI w.r.t. true labels", None),
    ("ARI_rows",         "ARI",     "ARI w.r.t. row clusters", None),
    ("ARI_cols",         "ARI",     "ARI w.r.t. column clusters", None),
]

for metric, metric_label, title, range_values in metrics:
    
    plot_heatmap(
        df=taucc_fair,
        metric=metric,
        algorithm="taucc_fair",
        dataset=dataset,
        sensitive=sensitive,
        plot_path=heatmap_path,
        metric_label=metric_label,
        title=title,
        range_values=range_values
    )

    plot_heatmap(
        df=taucc_fair_max,
        metric=metric,
        algorithm="taucc_fair_max",
        dataset=dataset,
        sensitive=sensitive,
        plot_path=heatmap_path,
        metric_label=metric_label,
        title=title,
        range_values=range_values
    )

## scatter plot 

In [ ]:
#df_v2 = taucc_fair_max.query("row_fair_majority==1.0 and row_fair_minority==1.0 and col_fair_majority==1.0 and col_fair_minority==1.0")

In [ ]:
if groups == 2:
    df_v1 = taucc_fair.query("fair_majority==1.0 and fair_minority==1.0")
    df_v2 = taucc_fair_max.query("fair_majority==1.0 and fair_minority==1.0")
else:
    df_v1 = taucc_fair.query("fair_majority==1.0 and fair_minority1==1.0 and fair_minority2==1.0")
    df_v2 = taucc_fair_max.query("fair_majority==1.0 and fair_minority1==1.0 and fair_minority2==1.0")

In [ ]:
markers = {
    "Fast-$\\tau$CC": "o",
    "Fair-$\\tau$CC v1": "s",
    "Fair-$\\tau$CC v2": "^"
}

In [ ]:
plt.figure(figsize=(7, 6))

vmin, vmax = 0.0, 1.0
sc = None

for algo in ["Fast-$\\tau$CC", "Fair-$\\tau$CC v1", "Fair-$\\tau$CC v2"]:
#for algo in ["Fair-$\\tau$CC v2"]:
    
    if algo == "Fast-$\\tau$CC":
        sub = taucc_vanilla.sort_values("sensitive_p")
    elif algo == "Fair-$\\tau$CC v1":
        sub = df_v1.sort_values("sensitive_p")
    elif algo == "Fair-$\\tau$CC v2":
        sub = df_v2.sort_values("sensitive_p")

    if sub.empty:
        continue

    x = sub["tau_x_mean"].values
    y = sub["tau_y_mean"].values
    balance = sub["balance_bera_mean"].values
    sensitive = sub["sensitive_p"].values

    # scatter
    sc = plt.scatter(
        x,
        y,
        c=balance,
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
        s=90,
        marker=markers.get(algo, "o"),
        edgecolor="black",
        linewidth=0.6,
        label=algo,
        zorder=3
    )

    # traiettoria al variare di sensitive_p
    plt.plot(
        x,
        y,
        linestyle="-",
        linewidth=1.2,
        alpha=0.7,
        zorder=2
    )

    # annotazioni per alcuni valori chiave di sensitive_p
    for xi, yi, sp in zip(x, y, sensitive):
        if sp in [0.0, 0.5, 1.0]:
            plt.annotate(
                f"{sp:.1f}",
                (xi, yi),
                textcoords="offset points",
                xytext=(4, 4),
                fontsize=8,
                alpha=0.85
            )

# colorbar
cbar = plt.colorbar(sc)
cbar.set_label("Balance", fontsize=12)

# assi
plt.xlabel("tau_x", fontsize=12)
plt.ylabel("tau_y", fontsize=12)

plt.grid(True, linestyle="--", alpha=0.3)
plt.legend(frameon=True, fontsize=10)
plt.tight_layout()
#plt.savefig(heatmap_path + "/taucc_fair_v2_scatter.png", dpi=300, bbox_inches="tight")
plt.show()

# Old code

Balance

In [ ]:
# Colori adatti per paper (palette colorblind-friendly)
COLOR_VANILLA   = "#E69F00"  # arancione
COLOR_FAIR      = "#0072B2"  # blu scuro
COLOR_FAIR_MAX  = "#009E73"  # verde teal

# Vanilla TauCC (fisso, retta orizzontale)
vanilla_bera_mean = np.full(len(x), taucc_vanilla["balance_bera_mean"].values[0])
vanilla_bera_std  = np.full(len(x), taucc_vanilla["balance_bera_std"].values[0])

# Fair TauCC
balance_bera_mean_fair     = np.array(taucc_fair["balance_bera_mean"].values)
balance_bera_std_fair      = np.array(taucc_fair["balance_bera_std"].values)

# Fair Max TauCC
balance_bera_mean_fair_max = np.array(taucc_fair_max["balance_bera_mean"].values)
balance_bera_std_fair_max  = np.array(taucc_fair_max["balance_bera_std"].values)

plt.plot(x, vanilla_bera_mean, label='Fast $\\tau$CC', color=COLOR_VANILLA, linestyle='--', linewidth=1.5)
plt.fill_between(x, vanilla_bera_mean - vanilla_bera_std, vanilla_bera_mean + vanilla_bera_std, alpha=0.15, color=COLOR_VANILLA)

plt.plot(x, balance_bera_mean_fair, label='Fair $\\tau$CC v1', color=COLOR_FAIR, linewidth=1.5)
plt.fill_between(x, balance_bera_mean_fair - balance_bera_std_fair, balance_bera_mean_fair + balance_bera_std_fair, alpha=0.15, color=COLOR_FAIR)

plt.plot(x, balance_bera_mean_fair_max, label='Fair $\\tau$CC v2', color=COLOR_FAIR_MAX, linewidth=1.5)
plt.fill_between(x, balance_bera_mean_fair_max - balance_bera_std_fair_max, balance_bera_mean_fair_max + balance_bera_std_fair_max, alpha=0.15, color=COLOR_FAIR_MAX)

plt.ylim(0.0, 1.0)
plt.xlim(0.0, 1.0)
plt.legend(framealpha=0.9, edgecolor='gray', fontsize=10)
plt.xlabel('$\\alpha$ minority group')
plt.ylabel('balance')
plt.title(f'Balance \n $\\alpha$ majority group = {alpha}')
plt.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
plt.tight_layout()
#plt.savefig(plot_path + f"/{dataset}_fairness.png", dpi=300)
plt.show()

In [ ]:
"""
# Vanilla TauCC
vanilla_bera_mean = np.full(len(x), df_vanilla["balance_bera_mean"].values[0])
vanilla_bera_std = np.full(len(x), df_vanilla["balance_bera_std"].values[0])

# Fair TauCC
balance_bera_mean = np.array(df["balance_bera_mean"].values)
balance_bera_std = np.array(df["balance_bera_std"].values)

plt.plot(x, balance_bera_mean, label='fair')
plt.fill_between(x, balance_bera_mean - balance_bera_std, balance_bera_mean + balance_bera_std, alpha=0.2, color='g')
plt.plot(x, vanilla_bera_mean, label='vanilla')
plt.fill_between(x, vanilla_bera_mean - vanilla_bera_std, vanilla_bera_mean + vanilla_bera_std, alpha=0.2)
plt.ylim(0.0, 1.0)
plt.xlim(0.0, 1.0)
plt.legend()
plt.xlabel('$\\alpha$ minority group')
plt.ylabel('balance')
plt.title(f'Balance [Bera et al.] \n $\\alpha$ majority group = {alpha}')
plt.grid(True)
plt.tight_layout()
#plt.show()
plt.savefig(plot_path + f"/{dataset}_fairness.png", dpi=300)
"""

ARI w.r.t. true labels

In [ ]:
vanilla_ARI_mean = np.full(len(x), df_vanilla["ARI_mean"].values[0])
vanilla_ARI_std = np.full(len(x), df_vanilla["ARI_std"].values[0])

ARI_mean = np.array(df["ARI_true_labels_mean"].values)
ARI_std = np.array(df["ARI_true_labels_std"].values)

plt.plot(x, ARI_mean, label='fair')
plt.fill_between(x, ARI_mean - ARI_std, ARI_mean + ARI_std, alpha=0.2, color='g')
plt.plot(x, vanilla_ARI_mean, label='vanilla')
plt.fill_between(x, vanilla_ARI_mean - vanilla_ARI_std, vanilla_ARI_mean + vanilla_ARI_std, alpha=0.2)
plt.ylim(0.0, 0.15)
plt.xlim(0.0, 1.0)
plt.legend()
plt.xlabel('$\\alpha$ minority group')
plt.ylabel('ARI')
plt.title(f'ARI w.r.t. true labels \n $\\alpha$ majority group = {alpha}')
plt.grid(True)
plt.tight_layout()
#plt.show()
plt.savefig(plot_path + f"/{dataset}_ARI_true.png", dpi=300)

ARI w.r.t. vanilla TauCC

In [ ]:
ARIrows_mean = np.array(df["ARI_rows_mean"].values)
ARIrows_std = np.array(df["ARI_rows_std"].values)
ARI_mean = np.array(df["ARI_cols_mean"].values)
ARI_std = np.array(df["ARI_cols_std"].values)

plt.plot(x, ARIrows_mean, label='ARI w.r.t. rows', color="green")
plt.fill_between(x, ARIrows_mean - ARIrows_std, ARIrows_mean + ARIrows_std, alpha=0.2, color="green")
plt.plot(x, ARI_mean, label='ARI w.r.t. cols')
plt.fill_between(x, ARI_mean - ARI_std, ARI_mean + ARI_std, alpha=0.2)
plt.ylim(-1.0, 1.0)
plt.xlim(0.0, 1.0)
plt.legend()
plt.xlabel('$\\alpha$ minority group')
plt.ylabel('ARI')
plt.title(f'ARI w.r.t. $\\tau$CC rows and columns \n $\\alpha$ majority group = {alpha}')
plt.grid(True)
plt.tight_layout()
#plt.show()
plt.savefig(plot_path + f"/{dataset}_ARI.png", dpi=300)

tau x and tau y

In [ ]:
vanilla_taux_mean = np.full(len(x), df_vanilla["tau_x_mean"].values[0])
vanilla_taux_std = np.full(len(x), df_vanilla["tau_x_std"].values[0])
vanilla_tauy_mean = np.full(len(x), df_vanilla["tau_y_mean"].values[0])
vanilla_tauy_std = np.full(len(x), df_vanilla["tau_y_std"].values[0])

taux_mean = np.array(df["tau_x_mean"].values)
taux_std = np.array(df["tau_x_std"].values)
tauy_mean = np.array(df["tau_y_mean"].values)
tauy_std = np.array(df["tau_y_std"].values)

In [ ]:
# fair
plt.plot(x, taux_mean, label='tau x (fair)')
plt.fill_between(x, taux_mean - taux_std, taux_mean + taux_std, alpha=0.2)
plt.plot(x, tauy_mean, label='tau y (fair)')
plt.fill_between(x, tauy_mean - tauy_std, tauy_mean + tauy_std, alpha=0.2)

# vanilla
plt.plot(x, vanilla_taux_mean, label='tau x (vanilla)')
plt.fill_between(x, vanilla_taux_mean - vanilla_taux_std, vanilla_taux_mean + vanilla_taux_std, alpha=0.2)
plt.plot(x, vanilla_tauy_mean, label='tau y (vanilla)')
plt.fill_between(x, vanilla_tauy_mean - vanilla_tauy_std, vanilla_tauy_mean + vanilla_tauy_std, alpha=0.2) 

plt.legend()
plt.xlabel('$\\alpha$ minority group')
plt.ylabel('tau_x, tau_y')
plt.title(f'tau_x and tau_y \n $\\alpha$ majority group = {alpha}')

plt.xlim(0.0, 1.0)

plt.grid(True)
plt.tight_layout()
#plt.show()
plt.savefig(plot_path + f"/{dataset}_tau.png", dpi=300)